[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/xpbd/xpbd.ipynb)

# A Chain of Rigid Links, by Position-Based Dynamics

This example lets a chain of ten rigid links, each a solid ellipsoid with three different axes, hang from a pin and fall from a horizontal arc, held together at its joints. Being extended bodies, the links do not only move but turn and twist about their own axes as the chain whips around. Each step lets every link move freely, then pulls the joints closed again: wherever two links have drifted apart, both are nudged back along the line between their anchors, each by as much as its inertia allows. This is extended position-based dynamics, the method of game physics, and it keeps the joints closed to a tiny fraction of a link.

A link's pose is a motor and its rate a bivector; its inertia maps rates to forques, lines of force. The gap at a joint is the line joining the two anchors, their regressive product: its norm is how far apart they are, its direction the line the correcting load acts along. Pulled into a link's frame, that forque turns through the inverse inertia into the twist it would cause, and the twist paired with the forque, `twist & forque`, is the link's compliance along the line. One multiplier closes the gap, and each link moves by the exponential of its twist.

In matrix notation the projection reads as explicit Jacobian matrices, a generalized inverse-mass matrix built from them and the inverse mass matrix, a solve for Lagrange multipliers, and separate linear and angular impulses on velocities and quaternion poses.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from itertools import islice

import numpy as np

from numga import NumpyContext
from examples.animation import show_animation
from examples.mechanics.xpbd import render, scenarios
from numga import Extensor
from examples.mechanics.xpbd.core import Chain, Forque, InverseInertia, Joints, Motor, Point, Rate, Scalar, external_forque, ga, lie, post_integrate

np.set_printoptions(precision=4, suppress=True)

## 1. The chain

Each link is six unit masses on its axes, placed to give it the inertia of a solid ellipsoid, so its three moments of inertia differ. Neighbours are joined at their ends; the chain starts on a horizontal arc, each link its neighbour carried along x and turned about z, the powers of one motor. The pin is a body of its own with zero inverse inertia, an infinite mass, so nothing moves it, and the first link swings from it. The joints are split into two halves whose links do not overlap, so that each half relaxes in one batch.

In [ ]:
# The links, the spacing of their joints, the semi-axes of each link's ellipsoid, and the turn between
# neighbours along the starting arc; the joints' compliance, the damping and gravity.
n_links, spacing, curl = 10, 9e-2, 0.3
semi_axes = np.array([0.62, 0.38, 0.22]) * spacing
compliance, damping, gravity = 1e-9, 1e-3, 0.5

In [ ]:
initial, partitions = scenarios.chain(NumpyContext(ga), n_links, spacing, semi_axes, curl, compliance, damping, gravity)

## 2. One step

A step integrates every link freely by a Lie-group step under gravity and damping, then closes each half of the joints in turn, reads the rates back from the motors' change, and finally cancels the joints' relative velocities with impulses along them.

In [ ]:
def project_distance_constraint(
    motors: Motor, anchors: Point, inverse_inertia: InverseInertia, compliance: Scalar, dt: float,
) -> Motor:
    """Close the gap between the anchors of body pairs, motors and anchors of shape [2, ...]."""
    # 1. Transform anchor points to world frame via motor pushforward:
    world_anchors = motors >> anchors

    # 2. Join the two world points into a line:
    # In PGA, the regressive product (&) of two points is the line connecting them.
    # Its norm is the Euclidean distance between the anchors. A closed joint has a
    # vanishing line; the small offsets keep its direction and multiplier at zero.
    line = world_anchors[0] & world_anchors[1]
    magnitude = line.norm()
    direction = line / (magnitude + 1e-24)

    # 3. Pull the forque back into each body's local coordinate frame:
    local_dir = motors << direction

    # 4. Inverse inertia maps the forque to an impulsive twist step:
    steps = inverse_inertia(local_dir)

    # 5. Generalized inverse mass / compliance along the constraint line:
    # Pairing the twist with the forque gives the effective compliance:
    # `inverse_inertia(local_dir) & local_dir`
    inertial_compliances = steps & local_dir
    total_compliance = (compliance / (dt**2)) + inertial_compliances.sum(axis=0) + 1e-24

    # 6. XPBD Lagrange multiplier with body signs (+1 for body 0, -1 for body 1):
    multiplier = (magnitude / total_compliance) * np.array([[+1], [-1]])

    # 7. Equal and opposite impulse steps on body 0 and body 1:
    impulse = steps * multiplier

    # 8. Corrective motor displacement via Lie algebra exponential map (half-angle):
    return motors * (impulse * -0.5).exp()


def project_velocity_constraint(
    motors: Motor, rates: Rate, anchors: Point, inverse_inertia: InverseInertia,
) -> Rate:
    """Cancel the relative velocity of the anchors of body pairs, with an impulse along it."""
    # Relative anchor velocity in world coordinates:
    anchors_map = anchors & anchors.commutator(Rate)
    velocities = motors >> anchors_map(rates)
    forque = velocities[1] - velocities[0]
    magnitude = forque.norm().select[0]
    direction = forque / (magnitude + 1e-24)

    local_dir = motors << direction
    steps = inverse_inertia(local_dir)
    inertial_compliances = steps & local_dir
    total_compliance = inertial_compliances.sum(axis=0) + 1e-24
    multiplier = (magnitude / total_compliance) * np.array([[+1], [-1]])
    impulse = steps * multiplier
    return rates + impulse


def step(chain: Chain, partitions: list[Joints], dt: float) -> Chain:
    """Perform a single XPBD integration step on the chain."""
    def forque(motor: Motor, rate: Rate) -> Forque:
        return external_forque(motor, rate, chain.first_moment, chain.gravity, chain.damping)

    # 1. Unconstrained inertial pre-integration:
    motor, rate = lie.explicit_rk4(chain.motor, chain.rate, chain.inertia, chain.inverse_inertia, dt, forque)

    # 2. Relax position constraints (Gauss-Seidel over red-black partitions):
    for joints in partitions:
        pair = joints.bodies
        relaxed = project_distance_constraint(
            motor[pair], joints.anchors, chain.inverse_inertia[pair], joints.compliance, dt
        )
        motor = motor.at[pair].set(relaxed)

    # 3. Post-integration: recover rates from motor displacement:
    motor, rate = post_integrate(chain.motor, motor, dt)

    # 4. Resolve velocity constraints:
    for joints in partitions:
        pair = joints.bodies
        resolved = project_velocity_constraint(motor[pair], rate[pair], joints.anchors, chain.inverse_inertia[pair])
        rate = rate.at[pair].set(resolved)

    return chain._replace(motor=motor, rate=rate)


def advance(chain: Chain, partitions: list[Joints], substeps: int, dt: float) -> Chain:
    """Advance the chain by dt in equal substeps."""
    for _ in range(substeps):
        chain = step(chain, partitions, dt / substeps)
    return chain


def joint_gaps(motor: Motor, partitions: list[Joints]) -> Scalar:
    """The Euclidean separation of the two anchors of every joint: the norm of their join."""
    def gap(joints: Joints) -> Scalar:
        world_anchors = motor[joints.bodies] >> joints.anchors
        # The join of two points is a simple line: its norm has no pseudoscalar part.
        return (world_anchors[0] & world_anchors[1]).norm()

    return Extensor.concatenate([gap(joints) for joints in partitions])

## 3. The swing

A hundred steps of four substeps each let the chain fall from its arc and swing. The projection is stable at any step; the substeps decide how closed the joints stay, here to a few millimetres, inside the links' overlapping ends. Each link is ray traced as its ellipsoid placed by its motor, its alternate octants darker so that its turning shows, over a floor that catches the shadows.

In [ ]:
n_steps, n_substeps, dt, frame_duration = 100, 4, 0.04, 60

In [ ]:
history = list(islice(scenarios.states(initial, lambda state: advance(state, partitions, n_substeps, dt)), n_steps + 1))
motors, gaps = scenarios.trajectory(history, partitions)                       # [steps + 1, bodies] Motor, [steps + 1, joints] Scalar
show_animation(render.animate_chain(motors, semi_axes, spacing), "xpbd_chain", frame_duration)

In [ ]:
print(f"largest joint gap: {gaps.to_array().max():.2e} m, against a spacing of {spacing} m")

In [ ]:
# checks
# The joints stay closed to within a tenth of the spacing, and the pin never leaves the origin.
origin = NumpyContext(ga).multivector.w.dual()
assert gaps.to_array().max() < 0.1 * spacing
assert ((motors[:, 0] >> origin) & origin).norm().to_array().max() < 1e-6